In [11]:
import librosa
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

# ======================
# 路径（不变）
# ======================
INPUT_CSV = r"/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/netease_songs_with_id.csv"
AUDIO_FOLDER = r"/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/netease_downloads"
OUTPUT_CSV = r"/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/Audio_AIGC_FINAL_FAST.csv"

# ======================
# 1. 读取 + 匹配（你已经成功了）
# ======================
df = pd.read_csv(INPUT_CSV)
all_files = [f for f in os.listdir(AUDIO_FOLDER) if f.endswith(".mp3")]

file_data = []
for f in all_files:
    name = f[:-4]
    if " - " in name:
        artist, song = name.split(" - ", 1)
        file_data.append({"artist_name": artist, "song_name": song, "filename": f})

file_df = pd.DataFrame(file_data)
merged_df = pd.merge(df, file_df, on=["artist_name", "song_name"], how="inner")
merged_df["audio_path"] = merged_df["filename"].apply(lambda x: os.path.join(AUDIO_FOLDER, x))

print(f"✅ 成功匹配歌曲：{len(merged_df)} 首")
print("⏩ 开始超快提取特征（1分钟内跑完）...")

# ======================
# 2. 超快特征提取（只保留最快、最稳的3个特征）
# ======================
def fast_features(path):
    try:
        # 只加载 10 秒！超级快
        y, sr = librosa.load(path, sr=22050, duration=10, mono=True)
        rms = np.mean(librosa.feature.rms(y=y))
        zcr = np.mean(librosa.feature.zero_crossing_rate(y=y))
        flat = np.mean(librosa.feature.spectral_flatness(y=y))
        return [rms, zcr, flat]
    except:
        return None

# ======================
# 3. 批量跑
# ======================
features_list = []
valid_rows = []

for i, (_, row) in enumerate(merged_df.iterrows()):
    if i % 50 == 0:
        print(f"已处理：{i}/{len(merged_df)}")
    f = fast_features(row["audio_path"])
    if f:
        features_list.append(f)
        valid_rows.append(row)

# ======================
# 4. 输出 AIGC 分数
# ======================
X = np.array(features_list)
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(X)

from sklearn.ensemble import IsolationForest
model = IsolationForest(contamination=0.2, random_state=42)
model.fit(X)

scores = -model.score_samples(X)
scores = (scores - scores.min()) / (scores.max() - scores.min())

# ======================
# 5. 保存结果
# ======================
result = pd.DataFrame(valid_rows)
result["Audio_AIGC_Rate"] = np.round(scores, 3)
result = result[["music_genre", "artist_name", "song_name", "Audio_AIGC_Rate"]]
result.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\n🎉🎉🎉 全部完成！")
print(f"✅ 结果已保存到：{OUTPUT_CSV}")
print(f"✅ 有效歌曲：{len(result)} 首")

✅ 成功匹配歌曲：575 首
⏩ 开始超快提取特征（1分钟内跑完）...
已处理：0/575


已处理：50/575
已处理：100/575
已处理：150/575
已处理：200/575
已处理：250/575
已处理：300/575
已处理：350/575
已处理：400/575
已处理：450/575
已处理：500/575
已处理：550/575

🎉🎉🎉 全部完成！
✅ 结果已保存到：/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/Audio_AIGC_FINAL_FAST.csv
✅ 有效歌曲：575 首


In [11]:
import pandas as pd
import numpy as np
import os
import re
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings("ignore")

# ======================
# 路径（我全部帮你改对！无错版！）
# ======================
AUDIO_CSV = "/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/AIGC rate/Audio_AIGC rate.csv"
LYRIC_FOLDER = "/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/lyrics_folder"
OUTPUT_CSV = "/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/AIGC rate/Audio_AIGC rate_with_lyric.csv"

# ======================
# 1. 读取音频AI率文件
# ======================
df = pd.read_csv(AUDIO_CSV, encoding="utf-8-sig")

# ======================
# 2. 读取歌词文件并建立匹配
# ======================
lyric_map = {}
for filename in os.listdir(LYRIC_FOLDER):
    if filename.endswith(".txt"):
        filename_no_ext = filename.replace(".txt", "")
        if "_" in filename_no_ext:
            artist_part, song_part = filename_no_ext.split("_", 1)
            song_clean = re.sub(r"[《》\s]", "", song_part)
            key = f"{artist_part.strip()}-{song_clean.strip()}"
            
            with open(os.path.join(LYRIC_FOLDER, filename), "r", encoding="utf-8") as f:
                lyric = f.read()
            
            lyric = re.sub(r"\[.*?\]", "", lyric)
            lyric = re.sub(r"\s+", " ", lyric).strip()
            lyric_map[key] = lyric

# ======================
# 3. 提取歌词特征（和音频一样 3 个特征）
# ======================
def get_feat(text):
    try:
        words = text.split()
        if len(words) < 5:
            return None
        wd = sum(len(w) for w in words) / len(words)
        ur = len(set(words)) / len(words)
        sents = re.split(r"[，。！？；:]", text)
        sents = [s for s in sents if len(s.strip()) > 0]
        al = sum(len(s) for s in sents) / len(sents) if len(sents) > 0 else 0
        return [wd, ur, al]
    except:
        return None

# ======================
# 4. 批量提取
# ======================
feats = []
ids = []

for idx, row in df.iterrows():
    artist = str(row["artist_name"]).strip()
    song = re.sub(r"[《》\s]", "", str(row["song_name"])).strip()
    key = f"{artist}-{song}"
    if key in lyric_map:
        f = get_feat(lyric_map[key])
        if f:
            feats.append(f)
            ids.append(idx)

# ======================
# 5. 训练模型（和音频代码完全一样！）
# ======================
X = np.array(feats)
X = StandardScaler().fit_transform(X)

model = IsolationForest(contamination=0.2, random_state=42)
model.fit(X)

scores = -model.score_samples(X)
scores = (scores - scores.min()) / (scores.max() - scores.min())
scores = np.round(scores, 3)

# ======================
# 6. 写入 Lyrics_AIGC_Rate 列
# ======================
df["Lyrics_AIGC_Rate"] = 0.0
for idx, score in zip(ids, scores):
    df.loc[idx, "Lyrics_AIGC_Rate"] = score

# ======================
# 7. 保存
# ======================
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("✅ 全部完成！")
print("✅ 已生成：Lyrics_AIGC_Rate（歌词AI率）")
print("✅ 保存到：Audio_AIGC rate_with_lyric.csv")

✅ 全部完成！
✅ 已生成：Lyrics_AIGC_Rate（歌词AI率）
✅ 保存到：Audio_AIGC rate_with_lyric.csv


In [12]:
import pandas as pd

# ======================
# 你刚刚生成的文件
# ======================
INPUT_CSV  = "/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/AIGC rate/Audio_AIGC rate_with_lyric.csv"
OUTPUT_CSV = "/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/AIGC rate/Audio_AIGC rate_FINAL.csv"

# ======================
# 读取
# ======================
df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

# ======================
# 计算总 AI 渗透率
# ======================
df["Total_AI_Permeability"] = (
    df["Audio_AIGC_Rate"] * 0.5 +
    df["Lyrics_AIGC_Rate"] * 0.5
).round(3)

# ======================
# 保存最终版
# ======================
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

# ======================
# 完成！
# ======================
print("✅ ✅ ✅ 全部完成！你的论文数据搞定了！")
print("✅ 已生成：Total_AI_Permeability（总AI渗透率）")
print(f"✅ 最终文件：{OUTPUT_CSV}")

print("\n预览结果：")
print(df[["artist_name", "song_name", "Audio_AIGC_Rate", "Lyrics_AIGC_Rate", "Total_AI_Permeability"]].head(10))

✅ ✅ ✅ 全部完成！你的论文数据搞定了！
✅ 已生成：Total_AI_Permeability（总AI渗透率）
✅ 最终文件：/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/AIGC rate/Audio_AIGC rate_FINAL.csv

预览结果：
  artist_name song_name  Audio_AIGC_Rate  Lyrics_AIGC_Rate  \
0         郑润泽     Intro            0.154             0.000   
1         郑润泽     Outro            0.102             0.144   
2         郑润泽        任性            0.232             0.008   
3         郑润泽        是你            0.232             0.087   
4         郑润泽      只在今夜            0.232             0.009   
5         郑润泽    像晴天像雨天            0.232             0.014   
6         郑润泽        晚点            0.232             0.008   
7         郑润泽        倔强            0.232             0.008   
8         郑润泽       隐形人            0.232             0.009   
9         郑润泽     一切的一切            0.232             0.013   

   Total_AI_Permeability  
0                  0.077  
1                  0.123  
2                  0.120  
3                  0.160  
4           